# Memory Tools for Agents
> Define and test tools for AI agents to interact with the SemanticMemory system.

## NBDev Default Export
Set the default export module name for nbdev.

In [ ]:
#| default_exp memorytools

## NBDev Hide
Import necessary nbdev functions like showdoc.

In [ ]:
#| hide
from nbdev.showdoc import *

## Import Required Libraries
Import `SemanticMemory` from the memory module, `Chat`, `toolloop`, `models` from `claudette`, and other necessary libraries like `json`.

In [ ]:
#| export
import json
from cogitarelink.memory import SemanticMemory # Assuming 00_memory.ipynb exports SemanticMemory
from claudette import Chat, toolloop, models, tool
from typing import Dict, List, Union, Optional, Any # Add Any if not present

In [ ]:
# Define simple data for our tools to use directly
CONTEXT_DATA = {
    "manufacturing": {
        "vocabulary": "http://example.org/manufacturing/",
        "key_terms": ["components", "assembledAt", "qualityScore"],
        "scoped_contexts": ["components"],
        "parent": None
    },
    "components": {
        "vocabulary": "http://example.org/components/",
        "key_terms": ["material", "supplier"],
        "scoped_contexts": [],
        "parent": "manufacturing"
    },
    "supplier": {
        "vocabulary": "http://example.org/supplier/",
        "key_terms": ["certification", "contactPerson"],
        "scoped_contexts": ["certification"],
        "parent": None
    },
    "certification": {
        "vocabulary": "http://example.org/certification/",
        "key_terms": ["type", "validUntil"],
        "scoped_contexts": [],
        "parent": "supplier"
    }
}

# Define self-contained functions that don't rely on external objects
def get_context_list() -> list:
    """Get a list of all available contexts.
    
    Returns:
        List of context IDs
    """
    return list(CONTEXT_DATA.keys())

def get_context_details(context_id: str) -> dict:
    """Get details about a specific context.
    
    Args:
        context_id: ID of the context to retrieve
        
    Returns:
        Dictionary with context details
    """
    if context_id in CONTEXT_DATA:
        return CONTEXT_DATA[context_id]
    return {"error": "Context not found"}

def find_relevant_contexts(query: str) -> list:
    """Find contexts relevant to a query.
    
    Args:
        query: Query text to analyze
        
    Returns:
        List of relevant context IDs with scores
    """
    # Simple keyword matching
    relevance = {
        "manufacturing": 0,
        "components": 0,
        "supplier": 0,
        "certification": 0
    }
    
    terms = query.lower().split()
    
    if any(word in terms for word in ["manufacturing", "product", "production"]):
        relevance["manufacturing"] += 2
    
    if any(word in terms for word in ["component", "part", "material"]):
        relevance["components"] += 2
        relevance["manufacturing"] += 1
    
    if any(word in terms for word in ["supplier", "vendor", "provide"]):
        relevance["supplier"] += 2
    
    if any(word in terms for word in ["certification", "quality", "standard"]):
        relevance["certification"] += 2
        relevance["supplier"] += 1
    
    # Return contexts with relevance > 0, sorted by relevance
    return [{"id": k, "score": v} for k, v in 
            sorted(relevance.items(), key=lambda x: x[1], reverse=True) 
            if v > 0]

def get_reasoning_hints() -> list:
    """Get hints about how to reason with these contexts.
    
    Returns:
        List of reasoning hints
    """
    return [
        "Properties using vocabulary http://example.org/manufacturing/ follow that vocabulary's semantics.",
        "The term 'components' has its own scoped context within the manufacturing context.",
        "Properties using vocabulary http://example.org/components/ follow that vocabulary's semantics.",
        "Properties using vocabulary http://example.org/supplier/ follow that vocabulary's semantics.",
        "The term 'certification' has its own scoped context within the supplier context.",
        "Properties using vocabulary http://example.org/certification/ follow that vocabulary's semantics."
    ]

def think(thought: str) -> str:
    """Use this tool to think about the context information and its implications.
    
    Args:
        thought: The thought to record
        
    Returns:
        The same thought, unchanged
    """
    return thought

def test_self_contained_tools():
    """Test self-contained context tools with Claude"""
    model = models[1]  # claude-3-7-sonnet
    
    print(f"Using model: {model}")
    
    # Create a chat with the self-contained tools
    chat = Chat(model, tools=[
        get_context_list,
        get_context_details,
        find_relevant_contexts,
        get_reasoning_hints,
        think
    ])
    
    # Test query
    prompt = """
    I want to understand how component materials are represented in the manufacturing context.
    Use the available tools to analyze which contexts are relevant and how they relate to each other.
    """
    
    print("\nSending prompt to Claude...")
    
    try:
        response = chat.toolloop(prompt)
        print(f"\nClaude's response:\n{response}")
        return response
    except Exception as e:
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()
        return f"Error: {str(e)}"


In [ ]:
test_self_contained_tools()

Using model: claude-3-7-sonnet-20250219

Sending prompt to Claude...

Claude's response:
Message(id='msg_01LECfv9fp3q9Jm1HYvtqbXC', content=[TextBlock(citations=None, text='Based on my analysis, here\'s how component materials are represented in the manufacturing context:\n\n1. **Hierarchical Structure**:\n   - The "manufacturing" context is the parent context\n   - The "components" context is a scoped context within manufacturing\n   - This indicates that components (and their materials) are a subset of the broader manufacturing domain\n\n2. **Component Materials Representation**:\n   - Materials are represented as a key term within the "components" context\n   - The components context uses the vocabulary namespace "http://example.org/components/"\n   - Key terms in the components context include "material" and "supplier"\n\n3. **Semantic Relationships**:\n   - Manufacturing has key terms including "components", "assembledAt", and "qualityScore"\n   - This suggests that components (wh

Based on my analysis, here's how component materials are represented in the manufacturing context:

1. **Hierarchical Structure**:
   - The "manufacturing" context is the parent context
   - The "components" context is a scoped context within manufacturing
   - This indicates that components (and their materials) are a subset of the broader manufacturing domain

2. **Component Materials Representation**:
   - Materials are represented as a key term within the "components" context
   - The components context uses the vocabulary namespace "http://example.org/components/"
   - Key terms in the components context include "material" and "supplier"

3. **Semantic Relationships**:
   - Manufacturing has key terms including "components", "assembledAt", and "qualityScore"
   - This suggests that components (which include materials) are tracked in terms of where they're assembled and their quality
   - The supplier information is linked to components, indicating that material sourcing is tracked

4. **Vocabulary and Semantics**:
   - Each context follows its own vocabulary's semantics
   - Properties using http://example.org/manufacturing/ follow manufacturing semantics
   - Properties using http://example.org/components/ follow component semantics
   - This separation allows for precise definition of material properties within the components domain

This representation allows for tracking component materials throughout the manufacturing process, including their suppliers, assembly locations, and quality metrics, all within a structured semantic framework.

<details>

- id: `msg_01LECfv9fp3q9Jm1HYvtqbXC`
- content: `[{'citations': None, 'text': 'Based on my analysis, here\'s how component materials are represented in the manufacturing context:\n\n1. **Hierarchical Structure**:\n   - The "manufacturing" context is the parent context\n   - The "components" context is a scoped context within manufacturing\n   - This indicates that components (and their materials) are a subset of the broader manufacturing domain\n\n2. **Component Materials Representation**:\n   - Materials are represented as a key term within the "components" context\n   - The components context uses the vocabulary namespace "http://example.org/components/"\n   - Key terms in the components context include "material" and "supplier"\n\n3. **Semantic Relationships**:\n   - Manufacturing has key terms including "components", "assembledAt", and "qualityScore"\n   - This suggests that components (which include materials) are tracked in terms of where they\'re assembled and their quality\n   - The supplier information is linked to components, indicating that material sourcing is tracked\n\n4. **Vocabulary and Semantics**:\n   - Each context follows its own vocabulary\'s semantics\n   - Properties using http://example.org/manufacturing/ follow manufacturing semantics\n   - Properties using http://example.org/components/ follow component semantics\n   - This separation allows for precise definition of material properties within the components domain\n\nThis representation allows for tracking component materials throughout the manufacturing process, including their suppliers, assembly locations, and quality metrics, all within a structured semantic framework.', 'type': 'text'}]`
- model: `claude-3-7-sonnet-20250219`
- role: `assistant`
- stop_reason: `end_turn`
- stop_sequence: `None`
- type: `message`
- usage: `{'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 1694, 'output_tokens': 319}`

</details>

In [ ]:
#| export
# ... existing imports ...
from fastcore.basics import patch # Add patch import
import httpx # Needed for SemanticMemory client init if not implicitly imported
from datetime import datetime # Needed for _load_known_contexts if used there (though not in this version)
from rdflib import URIRef, Literal, Namespace, RDF, RDFS, Dataset # Needed for SemanticMemory internals

# Add this cell
#| export
@patch
def _load_known_contexts(self:SemanticMemory):
    """Load and register common/known contexts (patched for testing)."""
    print("Pre-loading known contexts (patched)...")

    # Define contexts relevant to supply chain and potentially Wikidata
    contexts_to_load = {
        "manufacturing": {
            "@context": {
                "@version": 1.1,
                "@vocab": "http://example.org/manufacturing/",
                "components": {
                    "@container": "@list",
                    "@context": {
                        "@vocab": "http://example.org/components/",
                        "material": "materialType",
                        "supplier": {"@type": "@id"}
                    }
                },
                "assembledAt": "productionFacility",
                "qualityScore": {"@type": "http://www.w3.org/2001/XMLSchema#decimal"}
            }
        },
        "logistics": {
             "@context": {
                "@version": 1.1,
                "@vocab": "http://example.org/logistics/",
                "shipment": {
                    "@context": {
                        "@vocab": "http://example.org/shipping/",
                        "status": "currentStatus",
                        "location": "currentLocation",
                        "estimatedArrival": {"@type": "http://www.w3.org/2001/XMLSchema#dateTime"}
                    }
                },
                "trackingNumber": "shipmentIdentifier"
            }
        },
        "retail": {
            "@context": {
                "@version": 1.1,
                "@vocab": "http://schema.org/",
                "inventory": {
                    "@context": {
                        "@vocab": "http://example.org/inventory/",
                        "level": "stockLevel",
                        "reorderPoint": "minimumStockLevel"
                    }
                },
                "price": {"@type": "http://schema.org/PriceSpecification"},
                "category": "category"
            }
        },
        "supplier": {
             "@context": {
                "@version": 1.1,
                "@vocab": "http://example.org/supplier/",
                "certification": {
                    "@context": {
                        "@vocab": "http://example.org/certification/",
                        "type": "certificationType",
                        "validUntil": {"@type": "http://www.w3.org/2001/XMLSchema#date"}
                    }
                },
                "contactPerson": "representativeName"
            }
        },
         "wikidata_basic": { # Add this for the Wikidata part
            "@context": {
                "wd": "http://www.wikidata.org/entity/",
                "wdt": "http://www.wikidata.org/prop/direct/",
                "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
                "schema": "http://schema.org/",
                "label": "rdfs:label",
                "description": "schema:description",
                "P31": "wdt:P31", # instance of
                "P279": "wdt:P279", # subclass of
            }
        }
    }

    for name, context_data in contexts_to_load.items():
        try:
            # Use _register_contexts to add them
            context_id = self._register_contexts(context_data)
            if context_id:
                self.context_registry[context_id]["source"] = f"preloaded:{name}"
                print(f"  Successfully pre-loaded context '{name}' with ID: {context_id}")
            else:
                 print(f"  Warning: Failed to get ID for preloaded context '{name}'")
        except Exception as e:
            print(f"  Warning: Failed to pre-load context '{name}': {e}")

    print("Finished pre-loading known contexts (patched).")

# ... rest of the notebook cells ...

In [ ]:
#| export
def extract_context_data(memory: SemanticMemory) -> dict:
    """Extract context data from SemanticMemory into simple data structures.
    
    Args:
        memory: SemanticMemory instance
        
    Returns:
        Dictionary of context data with simple types
    """
    contexts = memory.list_contexts()
    context_data = {}
    
    for context_id, info in contexts.items():
        try:
            formatted = memory.format_context_for_llm(context_id)
            context_data[context_id] = {
                "id": context_id,
                "vocabulary": formatted.get("vocabulary_base", ""),
                "key_terms": list(formatted.get("key_terms", {}).keys()),
                "parent": info.get("parent"),
                "has_scoped_contexts": info.get("has_scoped_contexts", False),
                "scoped_terms": info.get("scoped_context_terms", [])
            }
        except Exception as e:
            # Handle any errors gracefully
            print(f"Warning: Could not extract data for context {context_id}: {str(e)}")
            context_data[context_id] = {
                "id": context_id,
                "error": str(e)
            }
    
    return context_data


In [ ]:
#| export
def create_context_tools(memory: SemanticMemory) -> dict:
    """Create tool functions for working with context data.
    
    Args:
        memory: SemanticMemory instance
        
    Returns:
        Dictionary of tool functions
    """
    # Extract data first
    context_data = extract_context_data(memory)
    
    # Generate reasoning hints
    reasoning_hints = []
    for ctx_id, ctx in context_data.items():
        if "error" in ctx:
            continue
            
        reasoning_hints.append(f"Properties using vocabulary {ctx['vocabulary']} follow that vocabulary's semantics.")
        if ctx['has_scoped_contexts'] and ctx['scoped_terms']:
            for term in ctx['scoped_terms']:
                child_contexts = [c for c, info in context_data.items() 
                                 if "parent" in info and info['parent'] == ctx_id]
                for child in child_contexts:
                    reasoning_hints.append(f"The term '{term}' has its own scoped context with ID {child}.")
    
    # Create standalone tool functions
    def get_context_list() -> list:
        """Get a list of all available contexts.
        
        Returns:
            List of context IDs
        """
        return list(context_data.keys())
    
    def get_context_details(context_id: str) -> dict:
        """Get details about a specific context.
        
        Args:
            context_id: ID of the context to retrieve
            
        Returns:
            Dictionary with context details
        """
        if context_id in context_data:
            return context_data[context_id]
        return {"error": "Context not found"}
    
    def find_relevant_contexts(query: str) -> list:
        """Find contexts relevant to a query.
        
        Args:
            query: Query text to analyze
            
        Returns:
            List of relevant context IDs with scores
        """
        terms = query.lower().split()
        results = []
        
        for ctx_id, ctx in context_data.items():
            if "error" in ctx:
                continue
                
            score = 0
            # Match against vocabulary
            if any(term in ctx['vocabulary'].lower() for term in terms):
                score += 1
            # Match against key terms
            for key_term in ctx['key_terms']:
                if any(term in key_term.lower() for term in terms):
                    score += 2
            
            if score > 0:
                results.append({"id": ctx_id, "score": score})
        
        # Sort by score descending
        return sorted(results, key=lambda x: x["score"], reverse=True)
    
    def get_reasoning_hints() -> list:
        """Get hints about how to reason with these contexts.
        
        Returns:
            List of reasoning hints
        """
        return reasoning_hints
    
    def think(thought: str) -> str:
        """Use this tool to think about the context information and its implications.
        
        Args:
            thought: The thought to record
            
        Returns:
            The same thought, unchanged
        """
        return thought
    
    return {
        "get_context_list": get_context_list,
        "get_context_details": get_context_details,
        "find_relevant_contexts": find_relevant_contexts,
        "get_reasoning_hints": get_reasoning_hints,
        "think": think
    }


In [ ]:
#| export
def create_context_aware_chat(memory: SemanticMemory, model: str = "claude-3-7-sonnet-20250219") -> Chat:
    """Create a Claude chat with context tools.
    
    Args:
        memory: SemanticMemory instance
        model: Claude model to use
        
    Returns:
        Chat instance with context tools
    """
    # Create tool functions
    tools = create_context_tools(memory)
    
    # Create chat with all tools
    chat = Chat(model, tools=list(tools.values()))
    
    return chat


In [ ]:
#| export
def query_memory_with_claude(memory: SemanticMemory, query: str, model: str = "claude-3-7-sonnet-20250219"):
    """Query the memory using Claude with context tools.
    
    Args:
        memory: SemanticMemory instance
        query: Query text
        model: Claude model to use
        
    Returns:
        Claude's response
    """
    chat = create_context_aware_chat(memory, model)
    
    prompt = f"""
    You are an AI assistant with access to a semantic memory system that uses JSON-LD contexts.
    
    Your task is to:
    1. Identify which contexts are relevant to this query
    2. Understand how these contexts relate to each other
    3. Answer the query using information from these contexts
    
    Query: {query}
    """
    
    return chat.toolloop(prompt)


In [ ]:
# Example usage
def test_memorytools():
    """Test the memorytools module with a sample memory"""
    # Create a memory instance
    memory = SemanticMemory()
    
    # Add manufacturing context
    manufacturing_context = {
        "@version": 1.1,
        "@vocab": "http://example.org/manufacturing/",
        "components": {
            "@container": "@list",
            "@context": {
                "@vocab": "http://example.org/components/",
                "material": "materialType",
                "supplier": {"@type": "@id"}
            }
        }
    }
    
    # Add a product with this context
    product = {
        "@context": manufacturing_context,
        "@id": "http://example.org/product/x1",
        "productName": "Widget X1",
        "components": [
            {
                "name": "Frame",
                "material": "Aluminum"
            }
        ]
    }
    
    memory.add_jsonld(product)
    
    # Test extraction
    context_data = extract_context_data(memory)
    print("Extracted context data:")
    for ctx_id, ctx in context_data.items():
        print(f"  {ctx_id}: {ctx['vocabulary']}")
    
    # Test creating tools
    tools = create_context_tools(memory)
    print(f"\nCreated {len(tools)} tools")
    
    # Test a query
    query = "What materials are used in components?"
    print(f"\nQuery: {query}")
    
    try:
        response = query_memory_with_claude(memory, query)
        print(f"Response: {response}")
    except Exception as e:
        print(f"Error during query: {str(e)}")
    
    return memory

# Uncomment to run the test
# test_memorytools()


In [ ]:
test_memorytools()

Extracted context data:
  context:e99496384e7f9e645d89955bfe77dbca: http://example.org/manufacturing/
  context:0b128f19e60579c2e428fb6847eb1d16: http://example.org/components/

Created 5 tools

Query: What materials are used in components?
Response: Message(id='msg_015XCF9zSBAr3SDHXUUYj9mc', content=[TextBlock(citations=None, text='Based on the information from the semantic memory system, I can answer your query about materials used in components:\n\n## Answer: What materials are used in components?\n\nAccording to the semantic memory system, components have "material" as one of their key attributes. The information is organized in a hierarchical structure where:\n\n1. Components are defined within the manufacturing domain (http://example.org/manufacturing/)\n2. Materials are properties of components, defined in the components vocabulary (http://example.org/components/)\n\nWhile the specific list of materials isn\'t directly provided in the context information, the system recognizes t

<cogitarelink.memory.SemanticMemory>

In [ ]:
import dspy # Add dspy import

# Configure DSPy LM for this notebook's context
try:
    # Use the same model as in 01_retriever.ipynb
    lm = dspy.LM('anthropic/claude-3-5-sonnet-20241022')
    dspy.configure(lm=lm)
    print("DSPy configured successfully.")
except Exception as e:
    print(f"Error configuring DSPy: {e}")
    # Optionally raise the error or handle it if configuration is critical
    # raise e


/Users/cvardema/dev/git/LA3D/cogitarelink/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DSPy configured successfully.


In [ ]:
#| export
# ... existing imports ...
from cogitarelink.retriever import LODNavigator # Import the retriever

#| export
def test_memorytools_with_wikidata():
    """Test memorytools using data retrieved from Wikidata and supply chain examples."""
    print("\n=== Testing Memory Tools with Wikidata & Supply Chain Data ===\n")

    # --- Setup Phase ---
    # 1. Create SemanticMemory and Pre-load Contexts
    memory = SemanticMemory()
    try:
        memory._load_known_contexts() # Call the patched pre-loading method
    except AttributeError:
        print("Error: _load_known_contexts patch might not have been applied correctly.")
        return None
    except Exception as e:
        print(f"Error during context pre-loading: {e}")
        return None

    # 2. Add Supply Chain Data (copied from 00_memory.ipynb test_supply_chain_example)
    print("\nAdding Supply Chain data to SemanticMemory...")
    try:
        # Manufacturing context data
        manufacturing_context = {
            "@version": 1.1, "@vocab": "http://example.org/manufacturing/",
            "components": {"@container": "@list", "@context": {"@vocab": "http://example.org/components/", "material": "materialType", "supplier": {"@type": "@id"}}},
            "assembledAt": "productionFacility", "qualityScore": {"@type": "http://www.w3.org/2001/XMLSchema#decimal"}
        }
        smartphone_product = {
            "@context": manufacturing_context, "@id": "http://example.org/product/smartphone-x200", "@type": "Product",
            "productName": "Smartphone X200", "productionDate": "2023-10-15", "batchNumber": "SX200-2023-10-B42",
            "assembledAt": "Shenzhen Factory 7", "qualityScore": "9.8",
            "components": [
                {"@id": "http://example.org/component/screen-amoled", "name": "AMOLED Display", "material": "Glass-Composite", "partNumber": "SCR-X200-AM", "supplier": "http://example.org/supplier/displaytech"},
                {"@id": "http://example.org/component/battery-lithium", "name": "Lithium Battery Pack", "material": "Lithium-Ion", "partNumber": "BAT-5000mAh", "supplier": "http://example.org/supplier/powerplus"},
                {"@id": "http://example.org/component/processor-a12", "name": "A12 Processor", "material": "Silicon", "partNumber": "CPU-A12-3.1", "supplier": "http://example.org/supplier/chipworks"}
            ]
        }
        memory.add_jsonld(smartphone_product) # Use regular add_jsonld as it has context

        # Logistics context data
        logistics_context = {
            "@version": 1.1, "@vocab": "http://example.org/logistics/",
            "shipment": {"@context": {"@vocab": "http://example.org/shipping/", "status": "currentStatus", "location": "currentLocation", "estimatedArrival": {"@type": "http://www.w3.org/2001/XMLSchema#dateTime"}}},
            "trackingNumber": "shipmentIdentifier"
        }
        smartphone_shipment = {
            "@context": logistics_context, "@id": "http://example.org/shipment/sx200-batch-42", "@type": "Shipment",
            "shipmentDate": "2023-10-20", "contents": "Smartphone X200 - 1000 units", "origin": "Shenzhen Factory 7",
            "destination": "North American Distribution Center", "trackingNumber": "SHP-2023-10-42-NAM", "carrier": "Global Logistics Partners",
            "shipment": {"status": "In Transit", "location": "Pacific Ocean", "estimatedArrival": "2023-11-05T08:00:00Z", "mode": "Sea Freight", "container": "GLPC-98765"}
        }
        memory.add_jsonld(smartphone_shipment)

        # Add more supply chain data if needed (retail, supplier) following the pattern...
        print("Supply Chain data added successfully.")

    except Exception as e:
        print(f"Error adding Supply Chain data to memory: {e}")
        import traceback
        traceback.print_exc()
        # Decide if test should continue

    # 3. Fetch and Add Wikidata Data
    print("\nFetching and adding Wikidata data...")
    navigator = LODNavigator()
    wikidata_uri = "http://www.wikidata.org/entity/Q42" # Douglas Adams
    print(f"Fetching data for {wikidata_uri}...")
    retrieval_result = navigator.navigate(wikidata_uri)

    if not retrieval_result.get("success"):
        print(f"Failed to retrieve Wikidata data: {retrieval_result.get('error')}")
        # Decide if test should continue
    else:
        wikidata_jsonld = retrieval_result.get("json_ld")
        print(f"Successfully retrieved Wikidata data.")
        print("Adding Wikidata data to SemanticMemory...")
        try:
            # Use add_jsonld_with_graph if the result might be a graph structure
            if isinstance(wikidata_jsonld, dict) and '@graph' in wikidata_jsonld:
                 memory.add_jsonld_with_graph(wikidata_jsonld)
            elif wikidata_jsonld: # Ensure it's not None
                 memory.add_jsonld(wikidata_jsonld)
            else:
                 print("Warning: Retrieved Wikidata data was empty or None.")
            print("Wikidata data added successfully (or skipped if empty).")
        except Exception as e:
            print(f"Error adding Wikidata data to memory: {e}")
            import traceback
            traceback.print_exc()
            # Decide if test should continue

    # --- Query Phase ---
    # 4. Query with memorytools
    # Query focusing on Wikidata data, but context tools now know about supply chain contexts too
    query = "What is Douglas Adams (Q42) known for, based on the data? Also, list the available contexts."
    print(f"\nQuerying memory with Claude: '{query}'")

    try:
        response = query_memory_with_claude(memory, query)
        print(f"\nClaude's response:\n{response}")
        # Add assertions here if needed, e.g., check if response mentions Adams or contexts
        assert response is not None, "Claude response should not be None"
        # assert "Douglas Adams" in str(response), "Response should mention Douglas Adams" # Example assertion
        # assert "manufacturing" in str(response), "Response should mention manufacturing context" # Example assertion
        return response
    except Exception as e:
        print(f"Error during query_memory_with_claude: {e}")
        import traceback
        traceback.print_exc()
        return None

# ... rest of the notebook ...

In [ ]:
test_memorytools_with_wikidata()


=== Testing Memory Tools with Wikidata & Supply Chain Data ===

Pre-loading known contexts (patched)...
  Successfully pre-loaded context 'manufacturing' with ID: context:20de30c246474eabdebd3ab17cf788d2
  Successfully pre-loaded context 'logistics' with ID: context:eba5b00877103f96b9c4c711cb21542d
  Successfully pre-loaded context 'retail' with ID: context:6149af7dc8d667bf303e98259699be18
  Successfully pre-loaded context 'supplier' with ID: context:2f8b290d50bc68c6127a921c11a848a3
  Successfully pre-loaded context 'wikidata_basic' with ID: context:e4b3a08e3d2ffc258a12375ab3c4f90c
Finished pre-loading known contexts (patched).

Adding Supply Chain data to SemanticMemory...
Supply Chain data added successfully.

Fetching and adding Wikidata data...
Fetching data for http://www.wikidata.org/entity/Q42...
Successfully retrieved Wikidata data.
Adding Wikidata data to SemanticMemory...
Wikidata data added successfully (or skipped if empty).

Querying memory with Claude: 'What is Douglas A

ToolUseBlock(id='toolu_01VHG68HLX3wuiERo6F1QNzk', input={'context_id': 'context:2f8b290d50bc68c6127a921c11a848a3'}, name='get_context_details', type='tool_use')

<details>

- id: `msg_018ZTgNiwrLF4k3p7cEf9eZc`
- content: `[{'id': 'toolu_01VHG68HLX3wuiERo6F1QNzk', 'input': {'context_id': 'context:2f8b290d50bc68c6127a921c11a848a3'}, 'name': 'get_context_details', 'type': 'tool_use'}]`
- model: `claude-3-7-sonnet-20250219`
- role: `assistant`
- stop_reason: `tool_use`
- stop_sequence: `None`
- type: `message`
- usage: `{'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 2564, 'output_tokens': 79}`

</details>

## NBDev Export
Export the notebook cells to the library.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()